# Painel de Análise de Vendas com Visualização de Dados
**Apostila Python, páginas 67 e 68 (Praticando)**

Empresa varejista fictícia: consolidar dados de lojas e períodos, gerar relatórios e visualizar métricas de desempenho com **pandas, numpy, matplotlib e seaborn**.

## Etapa 1: Preparação e importação de dados
### 1) Bibliotecas

In [ ]:
# python -m pip install pandas numpy matplotlib seaborn
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### 2) Dataset `vendas.csv`
Como a apostila permite, crio um dataset fictício com as colunas pedidas (data, loja, produto, quantidade, preco_unitario) e mais duas que os gráficos da etapa 3 precisam: **regiao** (pizza) e **categoria** (boxplot). Coloco alguns valores ausentes de propósito para a etapa 5.

In [ ]:
rng = np.random.default_rng(42)

lojas = {"Loja Centro": "Sudeste", "Loja Savassi": "Sudeste", "Loja Recife": "Nordeste",
         "Loja Curitiba": "Sul", "Loja Goiânia": "Centro-Oeste", "Loja Manaus": "Norte"}
produtos = {
    "Notebook": ("Eletrônicos", 3500), "Celular": ("Eletrônicos", 2200), "Fone Bluetooth": ("Eletrônicos", 250),
    "Camiseta": ("Vestuário", 60), "Calça Jeans": ("Vestuário", 150), "Tênis": ("Vestuário", 320),
    "Cafeteira": ("Casa", 280), "Jogo de Panelas": ("Casa", 450), "Liquidificador": ("Casa", 190),
    "Livro": ("Papelaria", 55), "Caderno": ("Papelaria", 25),
}

n = 1200
datas = pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 365, n), unit="D")
loja = rng.choice(list(lojas), n, p=[0.24, 0.18, 0.17, 0.16, 0.13, 0.12])
produto = rng.choice(list(produtos), n)
quantidade = rng.integers(1, 8, n).astype(float)
preco = np.array([produtos[p][1] for p in produto]) * rng.uniform(0.85, 1.15, n)

vendas = pd.DataFrame({
    "data": datas.strftime("%Y-%m-%d"),
    "loja": loja,
    "regiao": [lojas[l] for l in loja],
    "produto": produto,
    "categoria": [produtos[p][0] for p in produto],
    "quantidade": quantidade,
    "preco_unitario": preco.round(2),
})
# alguns valores ausentes para tratar na etapa 5
vendas.loc[rng.choice(n, 25, replace=False), "quantidade"] = np.nan
vendas.loc[rng.choice(n, 25, replace=False), "preco_unitario"] = np.nan

vendas.to_csv("vendas.csv", index=False)

df = pd.read_csv("vendas.csv")
df["data"] = pd.to_datetime(df["data"])   # data como datetime
df.head()

### 3) Coluna faturamento

In [ ]:
df["faturamento"] = df["quantidade"] * df["preco_unitario"]
df.head()

### 4) Integridade do dataset

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

### 5) Tratar valores nulos com a média da coluna

In [ ]:
df["quantidade"] = df["quantidade"].fillna(round(df["quantidade"].mean()))
df["preco_unitario"] = df["preco_unitario"].fillna(df["preco_unitario"].mean())
df["faturamento"] = df["quantidade"] * df["preco_unitario"]   # recalcula sem nulos

df.isnull().sum()

## Etapa 2: Análise exploratória e estatísticas
### 6) Faturamento total por loja (maior para o menor)

In [ ]:
fat_loja = df.groupby("loja")["faturamento"].sum().sort_values(ascending=False)
fat_loja.round(2)

### 7) Ticket médio por loja

In [ ]:
ticket_medio = (df.groupby("loja")["faturamento"].sum() / df.groupby("loja").size()).sort_values(ascending=False)
ticket_medio.round(2)

### 8) Cinco produtos mais vendidos em quantidade

In [ ]:
top5 = df.groupby("produto")["quantidade"].sum().sort_values(ascending=False).head(5)
top5

### 9) Mês com maior e menor faturamento (Mês/Ano)

In [ ]:
fat_mensal = df.groupby(df["data"].dt.to_period("M"))["faturamento"].sum()
maior, menor = fat_mensal.idxmax(), fat_mensal.idxmin()
print(f"Maior faturamento: {maior.strftime('%m/%Y')} -> R$ {fat_mensal.max():,.2f}")
print(f"Menor faturamento: {menor.strftime('%m/%Y')} -> R$ {fat_mensal.min():,.2f}")

### 10) Coluna mês

In [ ]:
df["mes"] = df["data"].dt.month
resumo_mensal = df.groupby("mes")["faturamento"].sum()
resumo_mensal.round(2)

## Etapa 3: Visualizações com matplotlib e seaborn
(O tema global da etapa 4, item 17, já é aplicado aqui para valer em todos os gráficos.)

In [ ]:
# 17) tema global
sns.set_style("whitegrid")
sns.set_palette(sns.color_palette("deep"))
os.makedirs("graficos", exist_ok=True)

### 11) Barras: faturamento por loja

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
cores = sns.color_palette("deep", len(fat_loja))
barras = ax.bar(fat_loja.index, fat_loja.values / 1000, color=cores, label=fat_loja.index)
ax.bar_label(barras, fmt="%.0f mil", padding=3)
ax.set_title("Faturamento total por loja (2025)")
ax.set_xlabel("Loja")
ax.set_ylabel("Faturamento (R$ mil)")
ax.legend(title="Loja", fontsize=8)
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig("faturamento_por_loja.png", dpi=120)
plt.savefig("graficos/faturamento_por_loja.png", dpi=120)
plt.show()

### 12) Linhas: evolução mensal do faturamento

In [ ]:
nomes_meses = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun", "Jul", "Ago", "Set", "Out", "Nov", "Dez"]
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(resumo_mensal.index, resumo_mensal.values / 1000, marker="o", linewidth=2)
for x, y in zip(resumo_mensal.index, resumo_mensal.values / 1000):
    ax.annotate(f"{y:.0f}", (x, y), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=9)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(nomes_meses)
ax.set_title("Evolução mensal do faturamento em 2025")
ax.set_xlabel("Mês")
ax.set_ylabel("Faturamento (R$ mil)")
plt.tight_layout()
plt.savefig("graficos/evolucao_mensal.png", dpi=120)
plt.show()

### 13) Pizza: faturamento por região

In [ ]:
fat_regiao = df.groupby("regiao")["faturamento"].sum().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(fat_regiao, labels=fat_regiao.index, autopct="%1.1f%%", startangle=90,
       colors=sns.color_palette("deep", len(fat_regiao)), wedgeprops={"edgecolor": "white"})
ax.set_title("Distribuição do faturamento por região")
plt.tight_layout()
plt.savefig("graficos/faturamento_por_regiao.png", dpi=120)
plt.show()

### 14) Dispersão: quantidade x faturamento

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(data=df, x="quantidade", y="faturamento", hue="categoria", alpha=0.7, ax=ax)
# outliers: vendas acima do percentil 99
limite = df["faturamento"].quantile(0.99)
outliers = df[df["faturamento"] > limite]
ax.scatter(outliers["quantidade"], outliers["faturamento"], s=120, facecolors="none", edgecolors="red", label="Outliers (top 1%)")
ax.set_title("Quantidade vendida x faturamento por venda")
ax.set_xlabel("Quantidade de itens")
ax.set_ylabel("Faturamento da venda (R$)")
ax.legend(title="Categoria")
plt.tight_layout()
plt.savefig("graficos/dispersao_quantidade_faturamento.png", dpi=120)
plt.show()

### 15) Boxplot: preços por categoria

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
sns.boxplot(data=df, x="categoria", y="preco_unitario", hue="categoria", palette="deep", legend=True, ax=ax)
ax.set_yscale("log")   # escala log porque Eletrônicos tem preços muito maiores
ax.set_title("Distribuição dos preços unitários por categoria")
ax.set_xlabel("Categoria")
ax.set_ylabel("Preço unitário (R$, escala log)")
plt.tight_layout()
plt.savefig("graficos/boxplot_precos_categoria.png", dpi=120)
plt.show()

### 16) Heatmap: correlação entre variáveis numéricas

In [ ]:
corr = df[["quantidade", "preco_unitario", "faturamento", "mes"]].corr()
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax)
ax.set_title("Correlação entre as variáveis numéricas")
plt.tight_layout()
plt.savefig("graficos/heatmap_correlacao.png", dpi=120)
plt.show()

## Etapa 4: Estilização e apresentação
- **17)** Tema global `whitegrid` + paleta `deep` aplicado antes do primeiro gráfico.
- **18)** Todos os gráficos têm título descritivo e rótulos de eixos.
### 19) Layout com múltiplos gráficos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(fat_loja.index, fat_loja.values / 1000, color=sns.color_palette("deep", len(fat_loja)))
axes[0].set_title("Faturamento por loja")
axes[0].set_xlabel("Loja")
axes[0].set_ylabel("R$ mil")
axes[0].tick_params(axis="x", rotation=25)

axes[1].bar(fat_regiao.index, fat_regiao.values / 1000, color=sns.color_palette("deep", len(fat_regiao)))
axes[1].set_title("Faturamento por região")
axes[1].set_xlabel("Região")
axes[1].set_ylabel("R$ mil")

fig.suptitle("Comparação: faturamento por loja e por região", fontsize=14)
plt.tight_layout()
plt.savefig("graficos/comparacao_loja_regiao.png", dpi=120)
plt.show()

### 20) Exportação e relatório final

In [ ]:
resumo = pd.DataFrame({
    "faturamento_total": df.groupby("loja")["faturamento"].sum().round(2),
    "numero_vendas": df.groupby("loja").size(),
    "ticket_medio": ticket_medio.round(2),
    "itens_vendidos": df.groupby("loja")["quantidade"].sum(),
    "regiao": df.groupby("loja")["regiao"].first(),
}).sort_values("faturamento_total", ascending=False)
resumo["participacao_%"] = (resumo["faturamento_total"] / resumo["faturamento_total"].sum() * 100).round(1)

resumo.to_csv("resumo_analitico.csv", sep=";", encoding="utf-8-sig")
print("Gráficos salvos em graficos/:", sorted(os.listdir("graficos")))
resumo

In [ ]:
print(f"Faturamento total do ano: R$ {df['faturamento'].sum():,.2f}")
print(f"Loja líder: {fat_loja.index[0]} ({resumo['participacao_%'].iloc[0]}% do total)")
print(f"Região líder: {fat_regiao.index[0]} ({fat_regiao.iloc[0] / fat_regiao.sum() * 100:.1f}%)")
print(f"Produto mais vendido: {top5.index[0]} ({top5.iloc[0]:.0f} unidades)")
print(f"Correlação quantidade x faturamento: {corr.loc['quantidade', 'faturamento']:.2f}")
print(f"Correlação preço x faturamento: {corr.loc['preco_unitario', 'faturamento']:.2f}")

### Comentários sobre as visualizações
- **Lojas:** a Loja Centro lidera com folga (cerca de 29% do total), e as duas lojas do Sudeste juntas fazem dele a região mais forte (pizza, cerca de 46%). Recife e Manaus têm os menores faturamentos.
- **Ticket médio:** varia bastante. Curitiba tem o maior ticket (cerca de R$ 3,5 mil) e, mesmo com poucas vendas, fica em 2º lugar no faturamento. Recife tem o menor ticket (cerca de R$ 2,1 mil): vende bastante, mas vende itens mais baratos.
- **Evolução mensal:** o faturamento oscila de mês a mês sem uma tendência clara, já que os dados são simulados sem sazonalidade.
- **Dispersão e heatmap:** o faturamento depende muito mais do **preço** (correlação alta) do que da **quantidade**. As vendas de Notebook e Celular aparecem como *outliers* mesmo com poucas unidades.
- **Boxplot:** Eletrônicos tem preços muito maiores e com mais variação. Papelaria e Vestuário ficam concentrados em valores baixos.
- **Conclusão:** para aumentar o faturamento, o foco deve ser a venda de eletrônicos e as lojas com menos vendas, e não só a quantidade de itens.